# Compare labels with ground truth
The code in this notebook displays the values, that we labeled incorrectly. It outputs, what the original label is and what we classified it as and also if we did not detect anything.

In [3]:
import pandas as pd

# dataset_name = 'imdb'
# dataset_name = 'weather'
dataset_name = 'medical'

polluted_data = pd.read_csv(f"../datasets/{dataset_name}_subset1_group1_w_errors.csv")
labels = pd.read_csv(f"../datasets/{dataset_name}_subset1_group1_error_mappings.csv")
gt_labels = pd.read_csv(f"../ground_truth_datasets/{dataset_name}_subset1_group1_error_mappings.csv")

unequal_labels = labels != gt_labels
equal_labels = labels == gt_labels

label_values = polluted_data.copy()


label_values[equal_labels] = None
# Convert values to strings and prepend the label value
for col in polluted_data.columns:
    for idx in polluted_data.index:
        if unequal_labels.loc[idx, col]:
            detected_label_number = labels.loc[idx, col]
            gt_label_number = gt_labels.loc[idx, col]
            original_value = polluted_data.loc[idx, col]
            label_values.loc[idx, col] = f"GT: {gt_label_number} | DET: {detected_label_number} | {original_value}"

display(label_values)

,encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,...,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted,admission_type_desc,admission_source_desc,discharge_disposition_desc
0,None,None,None,None,None,None,GT: 4 | DET: 3 | 25,GT: 4 | DET: 0 | 6,None,None,...,None,None,None,None,None,None,GT: 0 | DET: 3 | NO,None,GT: 1 | DET: 0 | Physician Referral,None
1,None,None,None,None,None,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,GT: 2 | DET: 1 | Discharged to home]
2,None,None,None,None,None,None,None,None,None,None,...,None,None,None,None,None,None,GT: 0 | DET: 3 | NO,None,GT: 1 | DET: 3 | Emergency Rome,None
3,None,None,None,None,None,None,None,None,None,None,...,None,None,None,None,None,None,GT: 0 | DET: 3 | NO,None,None,None
4,None,None,None,None,None,None,None,None,None,None,...,None,None,None,None,None,None,GT: 3 | DET: 0 | No,GT: 2 | DET: 3 | Emerbency,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20349,None,None,None,None,None,None,GT: 4 | DET: 0 | 1,GT: 4 | DET: 0 | 3,None,None,...,None,None,None,None,GT: 4 | DET: 2 | Yes,GT: 4 | DET: 0 | No,GT: 0 | DET: 3 | NO,None,None,None
20350,None,None,None,None,None,None,None,None,None,None,...,None,None,None,None,GT: 4 | DET: 2 | Yes,GT: 4 | DET: 0 | No,None,None,None,None
20351,None,None,None,None,None,None,None,None,None,None,...,None,None,None,None,None,None,GT: 0 | DET: 3 | NO,None,GT: 1 | DET: 0 | Physician Referral,None
20352,None,None,None,None,None,None,GT: 4 | DET: 0 | 1,GT: 4 | DET: 0 | 3,None,None,...,None,None,None,None,GT: 4 | DET: 2 | Yes,GT: 4 | DET: 0 | No,None,None,GT: 2 | DET: 3 | Phyician Reeferral,None


### Output statistics

In [17]:
medical_labels = pd.read_csv(f"../datasets/medical_subset1_group1_error_mappings.csv")
medical_gt_labels = pd.read_csv(f"../ground_truth_datasets/medical_subset1_group1_error_mappings.csv")

weather_labels = pd.read_csv(f"../datasets/weather_subset1_group1_error_mappings.csv")
weather_gt_labels = pd.read_csv(f"../ground_truth_datasets/weather_subset1_group1_error_mappings.csv")

imdb_labels = pd.read_csv(f"../datasets/imdb_subset1_group1_error_mappings.csv")
imdb_gt_labels = pd.read_csv(f"../ground_truth_datasets/imdb_subset1_group1_error_mappings.csv")



from tabulate import tabulate
ERROR_TYPES = {
    'NO_ERROR': 0,
    'MISSPELLING': 1, 
    'TYPO': 2,
    'OCR': 3,
    'WORD_TRANSPOSITION': 4
}

def calculate_stats(labels, gt_labels):
    stats = []
    for error_name, error_value in ERROR_TYPES.items():
        if error_name == 'NO_ERROR':
            continue
            
        # True positives - where both labels and gt_labels have this error type
        tp = ((labels == error_value) & (gt_labels == error_value)).sum().sum()
        
        # False positives - where labels has this error type but gt_labels has 0 (no error)
        fp_0_based = ((labels == error_value) & (gt_labels == 0)).sum().sum()
        
        # False positives - we missed marking the error, but it was actually present
        fp_unlabeled = ((labels == 0) & (gt_labels == error_value)).sum().sum()
        
        # False positives - detected wrong error type and ground truth has an error
        fp_wrong_error_type = ((labels == error_value) & (gt_labels != error_value) & (gt_labels != 0)).sum().sum()
        
        # Total false positives
        fp = fp_0_based + fp_unlabeled + fp_wrong_error_type
        
        # False negatives - where gt_labels has this error type but labels missed it
        fn = ((labels != error_value) & (gt_labels == error_value)).sum().sum()
        
        # Calculate precision and recall
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0
        
        # Calculate correct detection percentage
        total_detected = (labels == error_value).sum().sum()
        correct_detection = tp / total_detected if total_detected > 0 else 0
        
        stats.append([
            f"{error_value} - {error_name}",
            tp,
            fp_0_based,
            fp_unlabeled,
            fp_wrong_error_type,
            f"{correct_detection:.2%}",
            f"{precision:.2%}",
            f"{recall:.2%}",
        ])
    
    return stats

# Calculate and display stats for each dataset
for dataset_name, labels, gt_labels in [
    ("IMDB", imdb_labels, imdb_gt_labels),
    ("Weather", weather_labels, weather_gt_labels),
    ("Medical", medical_labels, medical_gt_labels)
]:
    print(f"\n{dataset_name} Dataset Statistics:")
    print(tabulate(
        calculate_stats(labels, gt_labels),
        headers=["Error Type", "TP", "0-based FP", "Unlabeled FP", "Wrong Error Type FP", "% Correct Detection", "Precision", "Recall"],
        tablefmt="grid"
    ))




IMDB Dataset Statistics:
+------------------------+--------+--------------+----------------+-----------------------+-----------------------+-------------+----------+
| Error Type             |     TP |   0-based FP |   Unlabeled FP |   Wrong Error Type FP | % Correct Detection   | Precision   | Recall   |
+========================+========+==============+================+=======================+=======================+=============+==========+
| 1 - MISSPELLING        | 155177 |         2359 |          35509 |                  7531 | 94.01%                | 77.37%      | 72.17%   |
+------------------------+--------+--------------+----------------+-----------------------+-----------------------+-------------+----------+
| 2 - TYPO               | 179901 |        41690 |          30950 |                 21842 | 73.90%                | 65.57%      | 74.75%   |
+------------------------+--------+--------------+----------------+-----------------------+-----------------------+-------------